# Exploring Gmsh meshes with pyramids-mesh

**Gmsh** is the widely used open-source finite-element mesh generator, and its `.msh` file is a common
interchange format for the unstructured meshes that feed FE/FV solvers. A Gmsh mesh is a **planar
engineering mesh** — raw `(x, y)` coordinates in model units, with **no georeferencing**.

The format has three axes that a reader has to cope with, and this notebook tours all three on genuine
files drawn from independent open-source projects (the issue-#94 validation corpus):

- **version** — the legacy **v2.2** layout versus the current **v4.1** layout;
- **encoding** — **ASCII** versus **binary** (Gmsh writes binary by default for large meshes);
- **cell type** — triangles, quads, and 3-D tetrahedra.

We read a 111-triangle mesh, explore and visualise it, then prove the two things that matter most for a
format tour: a v4.1 file reads to *exactly* the same mesh as its v2.2 twin, and a binary file reads to the
same `MeshDataset` as an ASCII one. We finish on the files the driver **refuses** — 3-D volume cells and
2nd-order elements — and why declining them is the right behaviour.

## 1. Imports

`MeshDataset` is the object every driver hands back. Importing from `pyramids` also puts its vendored
GDAL/PROJ data in place, so nothing here is order-sensitive in a notebook — just import the names you use.
`UnsupportedMeshFormatError` is the exception the driver raises for a `.msh` it cannot read; we import it now
for the negative section at the end. `add_tiles` would add a web basemap *when a mesh is georeferenced* —
Gmsh meshes never are, so we only ever call it behind an `if ds.epsg` guard.

In [ ]:
from pyramids.netcdf.ugrid import MeshDataset, UnsupportedMeshFormatError
import numpy as np
import pandas as pd
from cleopatra.basemap.tiles import add_tiles

## 2. Locate the example files

This notebook lives in `docs/examples/real-data/`, so the shared `examples/data/real-data/` corpus is three
levels up. The Gmsh samples live under `gmsh/`, with filenames that encode each file's verified properties:
`gmsh_v<version>_<ascii|binary>_<cell>-<dim>__<source>.msh`.

Paths inside a notebook are always written **relative to the notebook itself**, and pyramids' `read_file`
accepts a `pathlib.Path` directly — we pass the `Path`, never `str(path)`.

In [ ]:
from pathlib import Path
DATA = Path("../../../examples/data/real-data")
tri_file = DATA / "gmsh" / "gmsh_v2.2_ascii_tri3-2d__meshio-insulated.msh"
tri_file, tri_file.exists()

## 3. Read the first mesh

We start with `gmsh_v2.2_ascii_tri3-2d__meshio-insulated.msh` — a genuine Gmsh **2.2 ASCII** mesh of
**111 triangles** over **67 nodes**, from the meshio test suite. `MeshDataset.read_file(path, driver="gmsh")`
dispatches the file to the `gmsh` driver, which parses it and hands back a `MeshDataset` — the same pyramids
type produced from a native UGRID NetCDF file. We name the driver explicitly because several formats share
the `.msh` suffix.

In [ ]:
ds = MeshDataset.read_file(tri_file, driver="gmsh")
type(ds).__name__

## 4. Choose the plotting backend

cleopatra switches matplotlib to a headless `Agg` backend on the first `read_file`. Immediately after that
first read we call cleopatra's own chooser once: it detects the notebook and selects the inline backend (and
would fall back to `Agg` in a plain script or in CI), so every figure below embeds natively with no manual
rasterising.

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

From here nothing in the API remembers the data came from a Gmsh file — it is a first-class pyramids object.

## 5. Explore the dataset

A `MeshDataset` exposes summary properties describing the whole mesh: `n_node` / `n_face` (size),
`crs` / `epsg` (georeferencing, if the format recorded any), `bounds` (spatial extent), and
`data_variable_names` (any attached fields).

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": ds.bounds,
    "data_variable_names": ds.data_variable_names,
})

**111 faces** on **67 nodes**. `crs`/`epsg` are `None` — Gmsh stores raw model coordinates with no CRS, so
reprojection and basemap tiles are off the table — and `data_variable_names` is empty: a mesh-generator file
carries geometry only, no solver fields. The `bounds` sit in the model's own (unitless) coordinates, roughly
±3 about the origin.

## 6. The underlying `Mesh2d` topology

`ds.mesh` is the `Mesh2d` holding the raw topology. The array that tells you the **cell type** is
`face_node_connectivity.data` — the `(n_face, k)` table of the node indices of each face. `k` is the number
of nodes per face: **3** for triangles, 4 for quads.

In [ ]:
mesh = ds.mesh
conn = mesh.face_node_connectivity.data
cx, cy = mesh.face_centroids
pd.Series({
    "connectivity_shape": conn.shape,
    "nodes_per_face (k)": conn.shape[1],
    "face_areas_shape": mesh.face_areas.shape,
    "face_centroids_shape": (cx.shape, cy.shape),
})

The connectivity shape is **`(111, 3)`** — exactly **three** nodes per face, so this is a triangular mesh.
That matters later: circumcenters are well-defined for triangles but not for quads.

## 7. The wireframe

The simplest visualisation needs no data — just draw the cell edges. `plot_outline()` renders the mesh
skeleton through cleopatra and is the fastest sanity check that a file read in correctly. The tile call is
guarded by `if ds.epsg`, so it is skipped for this un-georeferenced mesh.

In [ ]:
_g = ds.plot_outline()
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The wireframe is the mesh *as the solver sees it* — an unstructured grid of triangles, finer where the
geometry demands resolution. Seeing it is the clearest confirmation the file parsed correctly.

## 8. A data map: per-cell area

This mesh carries no field values, so we synthesise one. We compute each cell's area from the geometry and
attach it as a face-located variable with `with_variable(...)`, which returns a **new** dataset (the original
`ds` is untouched). Colouring by it produces the generic pyramids data-map that any per-cell field (depth,
velocity, …) would use.

In [ ]:
d2 = ds.with_variable("face_area", mesh.face_areas, location="face")
_g = d2.plot("face_area", cmap="viridis", colorbar=True, title="Per-cell area")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

Each triangle is shaded by its area: darker cells are smaller (the refined regions), brighter cells larger
(the coarse interior). This is the generic pyramids data-map path — any per-cell field would plot
identically. `summary()` gives the numeric companion to the colour map.

In [ ]:
d2.summary()

The single `face_area` row reports the triangle count and the min/max cell sizes, confirming the
refinement gradient the colour map shows.

## 9. GIS operations, for free

Because the mesh is a pyramids object, the pyramids GIS layer applies directly — none of these methods live in
`pyramids-mesh`, which only reads the file. We tour three.

### Faces as a `GeoDataFrame`

`to_geodataframe()` turns every face into a polygon row of a geopandas `GeoDataFrame` — the standard vector
structure for export to shapefile/GeoPackage or for spatial joins. Plotting it draws every face as a filled
polygon, a vector-native second view of the mesh.

In [ ]:
gdf = ds.to_geodataframe()
ax = gdf.plot(edgecolor="black", linewidth=0.2, facecolor="#cfe8ef")
if ds.epsg:
    add_tiles(ax, crs=ds.epsg)
gdf.head(3)

One triangle polygon per row (111 in all). The CRS is unset — inherited from the mesh's `None` — so you would
assign one before exporting to a shapefile.

### Bounding polygon

`bounding_polygon()` dissolves the triangles into the mesh's outer boundary — a single shapely `Polygon`.
Because this is a small planar mesh with no periodic seam, it dissolves cleanly.

In [ ]:
poly = ds.bounding_polygon()
pd.Series({"type": type(poly).__name__, "area": poly.area, "bounds": ds.bounds})

A single `Polygon` outlining the meshed domain — an ordinary shapely geometry, the gateway to the
shapely/geopandas ecosystem for clipping and spatial joins.

### Circumcenters

`circumcenters()` returns an `(n_face, 2)` array of per-cell circumcenter coordinates — the points a
finite-volume scheme places unknowns at. It is **defined only for triangular meshes**; this mesh is
triangular, so it works. We overlay the circumcenters on the wireframe to see them sit inside the cells.

In [ ]:
cc = ds.circumcenters()
_g = ds.plot_outline()
_g.ax.scatter(cc[:, 0], cc[:, 1], s=8, color="crimson", zorder=5)
pd.Series({"shape": cc.shape, "first": tuple(np.round(cc[0], 3))})

`(111, 2)` — one `(x, y)` circumcenter per triangle, one red dot per cell. That these derived quantities are
a single method call away is the whole point of turning the file into a pyramids object.

## 10. Version equivalence: v2.2 vs v4.1

The same insulated mesh ships in both the legacy **v2.2** layout we just read and the current **v4.1** layout.
The two files differ byte-for-byte — v4.1 reorganises `$Nodes`/`$Elements` into per-entity blocks — but they
describe one mesh. Reading the v4.1 file should reproduce the v2.2 result exactly: same node count, same face
count, and identical coordinates and connectivity.

In [ ]:
v41 = MeshDataset.read_file(
    DATA / "gmsh" / "gmsh_v4.1_ascii_tri3-2d__meshio-insulated.msh", driver="gmsh"
)
same = (
    v41.n_node == ds.n_node
    and v41.n_face == ds.n_face
    and np.array_equal(v41.mesh.node_x, ds.mesh.node_x)
    and np.array_equal(v41.mesh.node_y, ds.mesh.node_y)
    and np.array_equal(v41.mesh.face_node_connectivity.data,
                       ds.mesh.face_node_connectivity.data)
)
assert same, "v4.1 read must match the v2.2 read"
pd.Series({"v41_n_node": v41.n_node, "v41_n_face": v41.n_face, "identical_to_v2.2": same})

Same **67 nodes / 111 faces**, with coordinates and connectivity matching bit-for-bit. The version byte in
`$MeshFormat` picks the parser; the mesh you get back is the same object, so a v2.2 file and a v4.1 file are
interchangeable to everything downstream.

## 11. Encoding equivalence: ASCII vs binary

Gmsh writes **binary** `.msh` by default for larger meshes, so real-world files are frequently binary rather
than ASCII. The driver decodes both: it reads the `$MeshFormat` *file-type* flag and the endianness marker,
then unpacks the packed `$Nodes`/`$Elements` records. `gmsh_v4.1_binary_tri3-2d__firedrake-square.msh` is a
tiny binary v4.1 mesh (two triangles over a unit square) from the Firedrake suite — reading it should yield an
ordinary `MeshDataset`, no different from an ASCII read.

In [ ]:
binf = MeshDataset.read_file(
    DATA / "gmsh" / "gmsh_v4.1_binary_tri3-2d__firedrake-square.msh", driver="gmsh"
)
pd.Series({
    "type": type(binf).__name__,
    "n_node": binf.n_node,
    "n_face": binf.n_face,
    "conn_k": binf.mesh.face_node_connectivity.data.shape[1],
    "bounds": binf.bounds,
})

A binary file reads straight into the same `MeshDataset` type — **4 nodes / 2 triangles** on the unit square,
`k = 3` — with every operation in this notebook available on it. The ASCII-vs-binary distinction is invisible
once the file is read; a binary `.msh` straight from a solver drops in with no extra handling.

Binary also carries the large meshes. `gmsh_v4.1_binary_tet4-3d__pylith-subduction.msh` is the file named in
issue #94: a 3-D PyLith **tetrahedral** subduction mesh. Its volume cells are not 2-D faces, but the driver
reads the **surface triangles** that bound the volume — 8 874 of them over 8 330 nodes — so it still returns a
usable 2-D `MeshDataset`. The node elevations surface as a `node_z` data variable, and the coordinates are
large projected metres (though no CRS is recorded).

In [ ]:
tet = MeshDataset.read_file(
    DATA / "gmsh" / "gmsh_v4.1_binary_tet4-3d__pylith-subduction.msh", driver="gmsh"
)
pd.Series({
    "n_node": tet.n_node,
    "n_face": tet.n_face,
    "data_variable_names": tet.data_variable_names,
    "epsg": tet.epsg,
    "bounds": tuple(round(b, 1) for b in tet.bounds),
})

A big binary mesh reads cleanly: 8 330 nodes, 8 874 surface triangles, a `node_z` field, and a ±400 km extent
in planar metres with no CRS attached. This is the issue-#94 file, now read rather than refused.

## 12. A different cell type: quads

`gmsh_v2.2_ascii_quad4-2d__precice-solid.msh` is a **quadrilateral** mesh (45 quads over 64 nodes) from the
preCICE tutorials. It reads exactly like the triangle meshes — but its connectivity has **four** nodes per
face, and that changes what geometry is defined.

In [ ]:
quad = MeshDataset.read_file(
    DATA / "gmsh" / "gmsh_v2.2_ascii_quad4-2d__precice-solid.msh", driver="gmsh"
)
pd.Series({
    "n_node": quad.n_node,
    "n_face": quad.n_face,
    "conn_k": quad.mesh.face_node_connectivity.data.shape[1],
    "bounds": quad.bounds,
})

**`k = 4`** — a quad mesh. Circumcenters are a triangle-only notion (a quad has no single circumscribed
circle), so `circumcenters()` refuses it with a `ValueError` rather than returning a wrong answer. The
face centroids are always available as the general-purpose per-cell point instead.

In [ ]:
try:
    quad.circumcenters()
except ValueError as e:
    print("quad mesh -> circumcenters skipped:", e)

qx, qy = quad.mesh.face_centroids
print("face_centroids available instead, shape:", qx.shape, qy.shape)

The quad mesh exposes the same explore/plot/GIS surface as the triangle mesh; only the triangle-specific
`circumcenters()` steps aside, pointing you at `face_centroids`.

## 13. What the driver refuses, and why

The `gmsh` driver reads **2-D, first-order** meshes. Two kinds of `.msh` are declined by design, each raising
`UnsupportedMeshFormatError` (a subclass of pyramids' `FileFormatNotSupportedError`) that names the offending
element type rather than silently misreading the file:

- a **3-D volume mesh** (`gmsh_v2.2_ascii_hex8-3d__firedrake-cube.msh` — hexahedra, element type 5);
- a **2nd-order / quadratic** mesh (`gmsh_v4.1_ascii_tri6-2d-quadratic__skfem.msh` — 6-node triangles, element
  type 8), whose mid-edge nodes have no place in a linear face table.

We loop over both, catch the exception, and report the element type each one tripped on.

In [ ]:
import re

rejected = [
    "gmsh_v2.2_ascii_hex8-3d__firedrake-cube.msh",
    "gmsh_v4.1_ascii_tri6-2d-quadratic__skfem.msh",
]
rows = []
for name in rejected:
    try:
        MeshDataset.read_file(DATA / "gmsh" / name, driver="gmsh")
        rows.append({"file": name, "raised": "(none - read!)", "element_type": ""})
    except UnsupportedMeshFormatError as e:
        m = re.search(r"element type (\d+)", str(e))
        rows.append({"file": name, "raised": type(e).__name__,
                     "element_type": m.group(1) if m else "?"})
pd.DataFrame(rows).set_index("file")

Both are refused by the same `UnsupportedMeshFormatError`, each naming the element type it found — 5 for the
3-D hexahedra, 8 for the 6-node quadratic triangles. The rejection is driven by the element codes inside the
file, not by the filename.

> **Note — xarray interop.** pyramids-core exposes a `to_xarray()` bridge to the xarray ecosystem, but these
> notebooks deliberately stay within pyramids-core's own GDAL-based UGRID I/O and do not import `xarray` (nor
> `h5py`, `netCDF4`, or `scipy`) directly.

## Takeaway

- The `gmsh` driver reads a `.msh` finite-element mesh into a pyramids `MeshDataset` across all three axes that
  vary between real files: **v2.2 or v4.1**, **ASCII or binary**, **triangles or quads**.
- **Version is invisible downstream**: the v4.1 insulated mesh reads to the *identical* nodes, coordinates and
  connectivity as its v2.2 twin.
- **Encoding is invisible too**: a binary `.msh` reads to the same `MeshDataset` as an ASCII one — including the
  large issue-#94 PyLith file, whose 3-D tetrahedra are read through their 8 874 bounding surface triangles.
- **Topology decides the geometry**: the triangle meshes support `circumcenters()`; the quad mesh (`k = 4`)
  refuses it with a `ValueError` and offers `face_centroids` instead.
- **Coordinates are planar, never georeferenced** (`crs`/`epsg` are `None`), so reprojection and basemap tiles
  are always skipped — yet the full pyramids GIS layer (`to_geodataframe`, `bounding_polygon`, circumcenters)
  applies to the mesh for free.
- **Out-of-scope files are declined, not misread**: 3-D volume cells and 2nd-order elements each raise an
  `UnsupportedMeshFormatError` naming the element type that tripped it.